# 🌍 Hotel Reservation System 8: The Guest's Language and Sentiment

Guests write in their own language, and some of them are angry. The core system from
`03_hotel_compliance_checker.ipynb` (coordinator, compliance checker and reservation agent) was built and
prompted in English, and its compliance rules are English text. Instead of changing the agents, this notebook
puts two **nodes in front of the coordinator** that describe the guest's latest message in the graph state:
`translate_in` detects its language and replaces it with an English version, and `sentiment_analysis` scores its
tone with a word lexicon. The coordinator de-escalates when the message reads as negative, and `translate_out`
translates the final reply back into the guest's language.

## Learning Objectives
1. **Detect and translate with structured output** - One LLM call returns the language code *and* the English
   text as typed fields
2. **Guest attributes in state** - Why a global `user_language` breaks with two guests, and how per-thread state
   fields fix it
3. **Lexicon sentiment as a code node** - TextBlob polarity, and a de-escalation instruction only for negative
   messages
4. **One composed graph** - `translate_in → sentiment_analysis → conv_assistant … → translate_out`, with the
   node order, the reply language and the apology checked in code
5. **Limits and alternatives** - The lexicon on raw vs translated text, sarcasm, and (optional) local Hugging
   Face models

## Where this fits
`07_hotel_merging_and_conversation_memory.ipynb` merged agent outputs and summarized long conversations. Both
nodes added here use one technique: a step before the coordinator that writes guest attributes into state. They
are built together because the sentiment node only works on the translated text (Part 6 measures it).
`09_hotel_system_wide_error_handling.ipynb` then makes the system recover from failures.

## Prerequisites
- The core system from `03_hotel_compliance_checker.ipynb` (read it first; this notebook imports it from `hotel_common.py`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- No corpus downloads: TextBlob's `PatternAnalyzer` ships its lexicon inside the package
- Optional Part 7 only: `uv sync --group hf` (local Hugging Face models, a large download)

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY` and imports the building blocks, TextBlob and `hotel_common`. The next two
cells create the model and the same database, SQL tools and compliance retriever as
`03_hotel_compliance_checker.ipynb`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

import sqlite3
from datetime import date, timedelta
from typing import Literal

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from pydantic import BaseModel, Field
from textblob import TextBlob

import hotel_common as hc  # shared hotel building blocks, in this folder (see 01_hotel_environment_and_coordinator.ipynb)
from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: shared by the translators, the three agents and the apology judge
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# DATABASE, SQL TOOLS AND THE COMPLIANCE RETRIEVER
# ============================================================================
hc.init_hotel_db("hotel.db")
sql_tools = hc.make_sql_tools("hotel.db")  # 02's tools: run_query only reads, book_room is the only write
retrieve = hc.build_compliance_retriever()
rag_tools = [retrieve]
print(f"✅ {len(sql_tools)} SQL tools, 1 retrieval tool over {len(hc.COMPLIANCE_RULES)} rules")

---
## 🔤 Part 1: Detect and Translate With Structured Output

Asking a model "what language is this, and what does it say in English?" and then parsing its prose is fragile.
With `with_structured_output(GuestMessage)` the answer comes back as a Pydantic object with two typed fields:
- `language_code`: the ISO 639-1 code (`en`, `de`, `fr`, `es`, ...)
- `english_text`: the message in English (unchanged if it already is)

A second, smaller schema (`DetectedLanguage`) only detects. It is used later as an independent **checker** of
the replies, so the check does not reuse the call that produced the result.

In [ ]:
# ============================================================================
# STRUCTURED DETECTION + TRANSLATION
# ============================================================================
class GuestMessage(BaseModel):
    """The language a guest wrote in, and the message in English."""

    language_code: str = Field(description="ISO 639-1 code of the language the message is written in, e.g. en, de, fr, es")
    english_text: str = Field(description="The message translated into English. If it is already English, copy it unchanged.")


class DetectedLanguage(BaseModel):
    """Only the language of a text."""

    language_code: str = Field(description="ISO 639-1 code of the language the text is written in, e.g. en, de, fr, es")


INBOUND_PROMPT = SystemMessage(content="""You receive one message written by a hotel guest.
Identify the language it is written in and translate it into English.
Keep names, dates, numbers and room types exactly as written.""")

reader = llm.with_structured_output(GuestMessage)
detector = llm.with_structured_output(DetectedLanguage)


def normalise(code: str) -> str:
    return code.strip().lower()[:2]


def read_guest_message(text: str) -> GuestMessage:
    parsed = reader.invoke([INBOUND_PROMPT, HumanMessage(content=text)])
    parsed.language_code = normalise(parsed.language_code)
    return parsed


def language_of(text: str) -> str:
    """Independent checker: which language is this text in?"""
    result = detector.invoke([SystemMessage(content="Identify the language of the text."), HumanMessage(content=text)])
    return normalise(result.language_code)


stay = date.today() + timedelta(days=60)
TEST_MESSAGES = [  # (expected language, what the guest writes)
    ("en", "Hi, is there parking at the hotel?"),
    ("de", "Hallo, wie lange ist die Rezeption heute geöffnet?"),
    ("fr", "Bonjour, avez-vous des chambres avec jacuzzi ?"),
    ("es", f"Hola, soy Dana Lee y quiero reservar una habitación Deluxe del {stay} al {stay + timedelta(days=2)}."),
]

llm_detected = {}
for expected, text in TEST_MESSAGES:
    parsed = read_guest_message(text)
    llm_detected[text] = parsed.language_code
    mark = "✅" if parsed.language_code == expected else "❌"
    print(f"{mark} expected {expected}  detected {parsed.language_code}  | {text}")
    print(f"      English: {parsed.english_text}\n")

**Reading the output:** each ✅ line means the detected code matched the language the message was written in.
Under it is the English version the agents will work with. In the Spanish booking request, check that the
name, the room type and both dates survived the translation unchanged: the reservation agent needs them exactly.

---
## 🧱 Part 2: Guest Attributes Belong in State

Both new nodes describe the guest's latest message: its language and its tone. A module-level variable such as
`user_language` would break as soon as two guests talk to the system: the last guest to write would set the
language for everyone. A **state field** is saved by the checkpointer per `thread_id`, so each conversation keeps
its own values.

`GuestState` adds five fields to the message list:
- `guest_language`: the ISO code of the guest's latest message
- `guest_text`: what the guest actually wrote (the message list holds the English version)
- `sentiment` and `polarity`: the lexicon's label and score for that message (Part 3)
- `reply`: the final reply, in the guest's language

The internal conversation (`messages`) stays in English, so the agents, the compliance rules, the SQL and the
sentiment lexicon all keep working unchanged.

In [ ]:
# ============================================================================
# THE STATE: WHAT THE NODES IN FRONT OF THE COORDINATOR WRITE
# ============================================================================
class GuestState(MessagesState):
    guest_language: str  # ISO 639-1 code of the guest's latest message
    guest_text: str      # the guest's message as written
    sentiment: str       # negative / neutral / positive, scored on the English text of that message
    polarity: float      # TextBlob polarity of that message, -1.0 .. +1.0
    reply: str           # the final reply, translated into guest_language


print("✅ GuestState fields:", list(GuestState.__annotations__))

---
## 📏 Part 3: Lexicon Sentiment as a Code Node

TextBlob's default analyser, `PatternAnalyzer`, looks every word up in a hand-built English lexicon
(`en-sentiment.xml`, shipped inside the package) and averages the scores of the words it knows, adjusting for
modifiers such as "very" and "not". It returns:
- **polarity**: −1.0 (negative) to +1.0 (positive)
- **subjectivity**: 0.0 (factual) to 1.0 (opinion)

`sentiment_assessments` also lists *which* words were scored, which is the key to understanding its mistakes.
`label()` turns polarity into three classes with two thresholds.

> TextBlob also offers a `NaiveBayesAnalyzer` trained on movie reviews. It needs NLTK corpora
> (`movie_reviews`, `punkt`) downloaded first and is just as English-only, so this notebook does not use it.

In [ ]:
# ============================================================================
# SCORE A FEW ENGLISH MESSAGES
# ============================================================================
NEGATIVE_BELOW = -0.1  # polarity below this -> negative
POSITIVE_ABOVE = 0.1   # polarity above this -> positive


def label(polarity: float) -> str:
    if polarity < NEGATIVE_BELOW:
        return "negative"
    if polarity > POSITIVE_ABOVE:
        return "positive"
    return "neutral"


def analyse(text: str) -> dict:
    result = TextBlob(text).sentiment_assessments
    scored = [" ".join(words) for words, *_ in result.assessments]
    return {"polarity": round(result.polarity, 2), "subjectivity": round(result.subjectivity, 2),
            "label": label(result.polarity), "scored_words": scored}


booking = date.today() + timedelta(days=60)
for text in [
    "I hate you",
    "Hey, your hotel is bad",
    f"Hey, I want to book a Family Suite from {booking} till {booking + timedelta(days=4)}",
]:
    r = analyse(text)
    print(f"{r['label']:<8} polarity={r['polarity']:+.2f} subjectivity={r['subjectivity']:.2f} "
          f"scored={r['scored_words'] or '-'} | {text}")

**Reading the output:** the first two messages score negative, and the `scored` list shows the single words
responsible. The booking request is factual: look at how few (if any) of its words are in the lexicon, which is
why it lands as neutral with low subjectivity.

### 3.1 ⚙️ The sentiment node, and what the coordinator does with it
The score is a pure function of the text, so sentiment needs no model call and no tool loop: it is a **plain
code node**, cheap, deterministic, and impossible to skip.
- `sentiment_analysis` runs right after `translate_in` on every turn and scores the **English** text in
  `messages`, writing `sentiment` and `polarity` into state. Part 6 shows why it must come after the translation.
- `conv_assistant` reads `state["sentiment"]` and, only when it is `negative`, adds a de-escalation instruction
  (`DEESCALATE`) to its prompt.

The cell tests the node on two hand-built states, with no model call.

In [ ]:
# ============================================================================
# THE SENTIMENT NODE AND THE DE-ESCALATION INSTRUCTION, TESTED WITHOUT A MODEL
# ============================================================================
def sentiment_analysis(state: GuestState) -> dict:
    r = analyse(state["messages"][-1].content)  # the guest's new message, already in English (translate_in ran first)
    return {"sentiment": r["label"], "polarity": r["polarity"]}


DEESCALATE = SystemMessage(content="""The guest's latest message reads as NEGATIVE.
Before anything else, acknowledge their frustration, apologise sincerely and ask how you can make it right.
Then continue with their request as usual.""")

for text, expected in [
    ("The room I had last week was dirty and the staff were rude. Terrible!", "negative"),
    ("I love your hotel, my last stay was wonderful!", "positive"),
]:
    update = sentiment_analysis({"messages": [HumanMessage(content=text)]})
    print(f"{'✅' if update['sentiment'] == expected else '❌'} {update} | {text}")

---
## 🔁 Part 4: The Composed Graph

`translate_in → sentiment_analysis → conv_assistant … → translate_out`
- **`translate_in`** (first node): reads the guest's message with `read_guest_message()`, stores the language
  and the original text in state, and **replaces** the message with its English version. Returning a
  `HumanMessage` with the same `id` makes the `add_messages` reducer overwrite it instead of appending.
- **`sentiment_analysis`** (Part 3) runs next, on that English text.
- **`conv_assistant`** is told to always answer in English, so nothing is translated twice, and it gets
  `DEESCALATE` only when `state["sentiment"] == "negative"`.
- **`translate_out`** (last node before `END`): translates the final English reply into `guest_language` and
  writes it to `reply`. For English guests it copies the reply without a model call.

The compliance checker and the reservation agent in between are `hc.make_core_nodes()`, unchanged. Every path
that used to end the turn now goes through `translate_out` first: the coordinator's reply, and the reservation
agent's reply after its tool loop. The last lines check that wiring in the compiled graph's edges, with no model
call.

In [ ]:
# ============================================================================
# CORE SPECIALISTS, TRANSLATE_IN/OUT, SENTIMENT-AWARE COORDINATOR, ROUTER AND GRAPH
# ============================================================================
sql_assistant, rag_assistant, generate = hc.make_core_nodes(llm, sql_tools, rag_tools)  # 03's agents, 04's verdict

OUTBOUND_PROMPT = """Translate the hotel assistant's reply below into the language with ISO 639-1 code '{code}'.
Keep names, dates, room numbers and prices unchanged. Return only the translation."""


def translate_in(state: GuestState) -> dict:
    guest_message = state["messages"][-1]
    parsed = read_guest_message(guest_message.content)
    return {
        "guest_language": parsed.language_code,
        "guest_text": guest_message.content,
        # same id -> add_messages replaces the guest's message with its English version
        "messages": [HumanMessage(content=parsed.english_text, id=guest_message.id)],
    }


def translate_out(state: GuestState) -> dict:
    english_reply = state["messages"][-1].content
    if state["guest_language"] == "en":
        return {"reply": english_reply}
    translated = llm.invoke([
        SystemMessage(content=OUTBOUND_PROMPT.format(code=state["guest_language"])),
        HumanMessage(content=english_reply),
    ])
    return {"reply": translated.content}


AGENTS = ["reservation_assistant", "compliance_checker"]
conversation_prompt = hc.conversation_prompt(
    AGENTS,
    extra_guidelines=hc.COMPLIANCE_FIRST + "\n"  # 03's guideline, plus:
    "- Always write in English. A translator converts your reply into the guest's language afterwards.",
)


def conv_assistant(state: GuestState) -> dict:
    extra = [DEESCALATE] if state.get("sentiment") == "negative" else []
    return {"messages": [llm.invoke([conversation_prompt, *extra] + state["messages"])]}


def choose_next_node(state: GuestState) -> Literal["reservation_assistant", "compliance_checker", "translate_out"]:
    return hc.route_by_mention(state["messages"], AGENTS) or "translate_out"  # 03's router, with its loop guard


builder = StateGraph(GuestState)
builder.add_node("translate_in", translate_in)
builder.add_node("sentiment_analysis", sentiment_analysis)
builder.add_node("conv_assistant", conv_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("compliance_checker", rag_assistant)
builder.add_node("generate", generate)
builder.add_node("sql_tools", ToolNode(sql_tools))
builder.add_node("rag_tools", ToolNode(rag_tools))
builder.add_node("translate_out", translate_out)

builder.add_edge(START, "translate_in")
builder.add_edge("translate_in", "sentiment_analysis")
builder.add_edge("sentiment_analysis", "conv_assistant")
builder.add_conditional_edges("conv_assistant", choose_next_node, ["reservation_assistant", "compliance_checker", "translate_out"])
builder.add_conditional_edges("reservation_assistant", tools_condition, {"tools": "sql_tools", END: "translate_out"})
builder.add_conditional_edges("compliance_checker", tools_condition, {"tools": "rag_tools", END: "conv_assistant"})
builder.add_edge("sql_tools", "reservation_assistant")
builder.add_edge("rag_tools", "generate")
builder.add_edge("generate", "conv_assistant")
builder.add_edge("translate_out", END)

graph = builder.compile(checkpointer=InMemorySaver())
print("✅ Graph compiled. Edges:")
hc.print_edges(graph)

# The wiring, checked from the compiled graph (no model call)
edges = {(edge.source, edge.target) for edge in graph.get_graph().edges}
front = [("__start__", "translate_in"), ("translate_in", "sentiment_analysis"), ("sentiment_analysis", "conv_assistant")]
into_end = sorted(source for source, target in edges if target == "__end__")
print(f"\n{'✅' if all(edge in edges for edge in front) else '❌'} START -> translate_in -> sentiment_analysis -> conv_assistant")
print(f"{'✅' if into_end == ['translate_out'] else '❌'} the only edge into __end__ comes from {into_end}")

**Reading the output:** the two ✅ lines are computed from the edge list above it. Every edge that used to reach
`__end__` now reaches `translate_out` first, so no reply can leave the system untranslated, and the sentiment
node sits between the translation and the coordinator.

---
## ▶️ Part 5: Four Languages, Then a Happy and an Angry German Guest

Each test message runs on its own thread. `ask()` runs one turn with `hc.run_traced()` and prints what the guest
wrote, the English the agents saw, the sentiment label, the nodes in the order they ran, the English reply and
the translated reply the guest receives. Then the independent `language_of()` checker detects the language of
each reply and compares it with the guest's.

In [ ]:
# ============================================================================
# ONE TURN PER LANGUAGE, THEN CHECK EACH REPLY'S LANGUAGE AND THE NODE ORDER
# ============================================================================
FRONT = ["translate_in", "sentiment_analysis", "conv_assistant"]


def ask(text: str, thread_id: str) -> tuple[dict, list[str]]:
    nodes, state = hc.run_traced(graph, text, thread_id, recursion_limit=30, verbose=False)
    english_request = next(m for m in reversed(state["messages"]) if m.type == "human").content
    print(f"👤 Guest [{state['guest_language']}]: {state['guest_text']}")
    print(f"   🇬🇧 agents saw:   {english_request}")
    print(f"   sentiment={state['sentiment']} (polarity {state['polarity']:+.2f}) | nodes: {' -> '.join(nodes)}")
    print(f"   🇬🇧 agents wrote: {state['messages'][-1].content}")
    print(f"🤵 Reply to guest: {state['reply']}\n")
    return state, nodes


runs = {}
for expected, text in TEST_MESSAGES:
    runs[expected] = ask(text, thread_id=f"lang-{expected}")
    print("=" * 80)

print("\n--- Checks (expected | stored in state | detected in the reply | node order) ---")
for expected, _ in TEST_MESSAGES:
    state, nodes = runs[expected]
    replied = language_of(state["reply"])
    order_ok = nodes[:3] == FRONT and nodes[-1] == "translate_out"
    mark = "✅" if expected == state["guest_language"] == replied and order_ok else "❌"
    print(f"{mark} {expected} | {state['guest_language']} | {replied} | {' -> '.join(nodes[:3])} ... {nodes[-1]}")

**Reading the output:** a ✅ row means four things agree: the language the guest wrote in, the
`guest_language` stored in that thread's state, the language a separate detector finds in the reply, and the
node order (`translate_in → sentiment_analysis → conv_assistant` first, `translate_out` last). The reply is checked
by a *different* call from the one that translated it, so a translator that silently answered in English would
show up as ❌.

### 5.1 🧾 Did the Spanish booking really happen?
The Spanish guest asked for a Deluxe room. The agents only ever saw the English version, so if the booking is in
the database, the translation carried the name and dates through intact.

In [ ]:
# ============================================================================
# CHECK THE DATABASE FOR THE SPANISH GUEST'S BOOKING
# ============================================================================
with sqlite3.connect("hotel.db") as conn:
    rows = pd.read_sql("SELECT * FROM reservations WHERE guest_name LIKE '%Dana%'", conn)
print(rows.to_string(index=False) if len(rows) else "No reservation for Dana Lee was written.")
print(f"\nBooking written with the requested dates: "
      f"{'✅' if len(rows) == 1 and rows.start_date[0] == str(stay) else '❌'}")

**Reading the output:** ✅ means one row for Dana Lee with the check-in date from the Spanish message. A ❌ with
no row usually means the coordinator asked a follow-up question (read its English reply above) rather than
forwarding the booking; the translation layer is not the cause if the English request above is complete.

### 5.2 😠 A happy and an angry German guest
Both guests ask an ordinary question in German; only the tone differs. The lexicon cannot read German, so the
sentiment label can only be right because `sentiment_analysis` scores the English version. The checks compare
the label with the expected one, print what the lexicon makes of the raw German text, and confirm that each
reply is in German.

In [ ]:
# ============================================================================
# SENTIMENT ON THE TRANSLATED TEXT, REPLY IN GERMAN
# ============================================================================
GERMAN_GUESTS = [  # (thread, message, expected sentiment)
    ("happy-de", "Ich liebe Ihr Hotel, mein letzter Aufenthalt war wunderbar! Haben Sie Zimmer mit Balkon?", "positive"),
    ("angry-de", "Das Zimmer letzte Woche war schmutzig und das Personal war unhöflich. Schrecklich! "
                 "Haben Sie Zimmer mit Whirlpool?", "negative"),
]
german_runs = {}
for thread, text, _ in GERMAN_GUESTS:
    german_runs[thread] = ask(text, thread_id=thread)
    print("=" * 80)

print("\n--- Checks ---")
for thread, text, expected in GERMAN_GUESTS:
    state, nodes = german_runs[thread]
    raw = analyse(text)["label"]  # what the lexicon makes of the German text itself
    print(f"{'✅' if state['sentiment'] == expected else '❌'} {thread}: labelled {state['sentiment']} on the English "
          f"text (expected {expected}); the raw German text scores {raw}")
    print(f"{'✅' if language_of(state['reply']) == 'de' else '❌'} {thread}: the reply is in German")

#### Did the coordinator actually de-escalate?
"The reply sounds apologetic" is a judgement, so a small judge makes it, but it does not answer yes/no. It must
**copy the apology sentence word for word** (or write `none`), and code decides: an apology counts only if the
copied sentence really appears in the reply. The judge reads the English reply (`messages[-1]`, before
`translate_out`). The angry guest should get an apology; the happy guest should not.

In [ ]:
# ============================================================================
# EVIDENCE-COPYING JUDGE FOR THE APOLOGY
# ============================================================================
class ApologyEvidence(BaseModel):
    apology_sentence: str = Field(
        description="Copy, word for word, the sentence in the reply that apologises to the guest or acknowledges "
        "their frustration. Write 'none' if there is no such sentence."
    )


judge = llm.with_structured_output(ApologyEvidence)


def apologises(reply: str) -> tuple[bool, str]:
    evidence = judge.invoke([
        SystemMessage(content="You inspect a hotel assistant's reply to a guest."),
        HumanMessage(content=f"Reply:\n{reply}"),
    ]).apology_sentence.strip()
    squash = lambda s: " ".join(s.split()).lower()  # noqa: E731
    return evidence.lower() != "none" and squash(evidence)[:40] in squash(reply), evidence


for thread, _, expected in GERMAN_GUESTS:
    state, _ = german_runs[thread]
    found, evidence = apologises(state["messages"][-1].content)  # the English reply, before translate_out
    should = expected == "negative"
    print(f"{'✅' if found == should else '❌'} {thread}: apology {'found' if found else 'absent'} "
          f"(expected {'yes' if should else 'no'}) | evidence: {evidence}")

**Reading the output:** two ✅ lines mean the de-escalation instruction changed the behaviour exactly where it
should: an apology for the angry guest, none for the happy one. The evidence column shows the sentence the
decision rests on. Because the instruction is added in code only when `state["sentiment"] == "negative"`, the
happy guest's coordinator never even saw it. On the raw German text the lexicon finds no word it knows (the
checks above print its label), so without the translation first the angry guest would get the breezy answer.

### 5.3 🧵 One language per conversation, updated every turn

Two properties a global variable cannot give:
1. **Isolation**: each thread keeps its own `guest_language`, read here straight from the checkpointer.
2. **Freshness**: the language is detected on *every* message. The German guest of thread `lang-de` now writes
   in English, and that thread switches to English for this reply, while the French thread is untouched.

In [ ]:
# ============================================================================
# PER-THREAD LANGUAGE FROM THE CHECKPOINTER
# ============================================================================
def stored_language(thread_id: str) -> str:
    return graph.get_state({"configurable": {"thread_id": thread_id}}).values["guest_language"]


print("Before:", {f"lang-{code}": stored_language(f"lang-{code}") for code, _ in TEST_MESSAGES})

state, _ = ask("Thanks! And do you serve breakfast?", thread_id="lang-de")

after = {f"lang-{code}": stored_language(f"lang-{code}") for code, _ in TEST_MESSAGES}
print("After: ", after)
print(f"\n{'✅' if after['lang-de'] == 'en' and language_of(state['reply']) == 'en' else '❌'} the German thread switched to English for an English message")
print(f"{'✅' if after['lang-fr'] == 'fr' else '❌'} the French thread still stores fr")

**Reading the output:** "Before" shows four threads with four languages at the same time, which one global
variable could never hold. "After" shows only `lang-de` changed. To chat with the system yourself, run
`hc.chat_with_agent(graph, "my-thread")` in a new cell: it prints the internal English transcript, and the
translated reply is in `graph.get_state(...).values["reply"]`.

---
## 🔬 Part 6: Lexicon Limits, on Raw and Translated Text

The lexicon is English, and it scores words, not meaning. The cell scores the same two reviews written in
English, German and French, plus one sarcastic English sentence, against the label a human would give. Every
German and French review is scored twice: on the raw text, and on the English that `read_guest_message()` (the
call inside `translate_in`) makes of it. Code counts how many of them each way gets right.

In [ ]:
# ============================================================================
# THE SAME REVIEWS, SCORED ON THE RAW TEXT AND ON THE TRANSLATION
# ============================================================================
SAMPLES = [  # (language, text, human label)
    ("en", "The service is terrible and the room was dirty.", "negative"),
    ("de", "Die Bedienung ist schrecklich und das Zimmer war schmutzig.", "negative"),
    ("fr", "Le service est horrible et la chambre était sale.", "negative"),
    ("en", "The staff were wonderful and the breakfast was delicious.", "positive"),
    ("de", "Das Personal war wunderbar und das Frühstück war köstlich.", "positive"),
    ("fr", "Le personnel était merveilleux et le petit-déjeuner délicieux.", "positive"),
    ("en", "Great, another cold shower. Just what I needed.", "negative"),  # sarcasm
]

raw_ok, translated_ok = [], []
for lang, text, human in SAMPLES:
    raw = analyse(text)
    print(f"{'✅' if raw['label'] == human else '❌'} {lang} human={human:<8} raw={raw['label']:<8} "
          f"({raw['polarity']:+.2f}) scored={raw['scored_words'] or '-'}\n      {text}")
    if lang != "en":
        english = read_guest_message(text).english_text  # what translate_in would hand to sentiment_analysis
        translated = analyse(english)
        print(f"   {'✅' if translated['label'] == human else '❌'} translated={translated['label']:<8} "
              f"({translated['polarity']:+.2f}) scored={translated['scored_words'] or '-'}\n      {english}")
        raw_ok.append(raw["label"] == human)
        translated_ok.append(translated["label"] == human)

sarcasm = analyse(SAMPLES[-1][1])
print(f"\n--- German and French reviews: raw {sum(raw_ok)}/{len(raw_ok)} correct, "
      f"translated first {sum(translated_ok)}/{len(translated_ok)} correct ---")
print(f"{'✅' if all(translated_ok) and sum(translated_ok) > sum(raw_ok) else '❌'} scoring the translation fixes "
      "the non-English reviews")
print(f"{'⚠️' if sarcasm['label'] != 'negative' else 'ℹ️'} the sarcastic sentence is labelled {sarcasm['label']}, "
      f"from the words {sarcasm['scored_words']}")

**Reading the output:** read the `scored` column first, because it explains every row:
- A raw German review scores `neutral` with `scored=-`: it found **no word** it knows. `schrecklich` and
  `wunderbar` are not in an English lexicon, so the polarity is 0.
- A raw French review that does get a score owes it to a word **spelt the same in English** (check which word is
  listed): a lucky cognate, not understanding. Change the wording and it disappears.
- The translated rows score the English words, which is why the ✅ count rises: that is the reason this
  notebook's graph runs `sentiment_analysis` **after** `translate_in`.
- The sarcastic English sentence is scored word by word: `great` counts as praise however it is meant (the ⚠️
  line). For sarcasm and mixed messages, an LLM classifier with structured output, like Part 1's detector, is the
  usual next step, at the cost of a model call.

---
## 🧪 Part 7 (Optional): Local Models With Hugging Face

The LLM does detection and translation in one call for any language pair. Local models can do the same job:
- **`papluca/xlm-roberta-base-language-detection`**: a classifier over 20 languages
- **`Helsinki-NLP/opus-mt-de-en`** and **`opus-mt-en-de`**: MarianMT translators, one model per direction

Local models cost nothing per call and keep guest text on your machine, but each language pair needs its own
model (roughly 300 MB each), and quality varies by pair.

> **This part does not run by default.** Its cells are tagged `skip-execution` because they need the optional
> dependency group and download several hundred MB of models. To run it: `uv sync --group hf` at the
> repository root, restart the kernel, run Step 1 and Part 1, then run these cells by hand. Without the group
> the first cell prints a message instead of failing.

In [ ]:
# ============================================================================
# OPTIONAL: LOAD THE LOCAL LANGUAGE DETECTOR (needs `uv sync --group hf`)
# ============================================================================
try:
    from transformers import MarianMTModel, MarianTokenizer, pipeline
except ImportError:  # the optional hf group is not installed
    MarianMTModel = MarianTokenizer = pipeline = None
    print("⚠️ transformers is not installed. Run `uv sync --group hf` at the repository root, then restart the kernel.")

DETECTOR_CKPT = "papluca/xlm-roberta-base-language-detection"
MARIAN_MODELS = {("de", "en"): "Helsinki-NLP/opus-mt-de-en", ("en", "de"): "Helsinki-NLP/opus-mt-en-de"}

if pipeline is not None:
    local_detector = pipeline("text-classification", model=DETECTOR_CKPT)
    print(f"✅ Loaded {DETECTOR_CKPT}")

`translation_service()` detects the language locally, then translates with MarianMT when a model exists for that
pair. `marian()` loads each model once (`lru_cache`), and a pair without a local model is reported plainly
instead of failing.

`top_label()` reads the detector's answer. Called on one string with `top_k=1`, the transformers 5.x
text-classification pipeline returns a list holding one `{"label", "score"}` dict (the 5.17 source:
`top_k` in the call switches off the legacy single-dict output). `top_label()` also accepts a bare dict or a
nested list, so a version that changes the nesting still works.

In [ ]:
# ============================================================================
# OPTIONAL: A LOCAL TRANSLATION SERVICE
# ============================================================================
from functools import lru_cache


@lru_cache(maxsize=None)
def marian(source: str, target: str):
    """Load (once) the MarianMT tokenizer and model for one direction."""
    name = MARIAN_MODELS[(source, target)]
    return MarianTokenizer.from_pretrained(name), MarianMTModel.from_pretrained(name)


def top_label(result) -> str:
    """The best label, whatever the nesting: [{...}] (transformers 5.x with top_k=1), [[{...}]] or a bare {...}."""
    while isinstance(result, list):
        result = result[0]
    return result["label"]


def local_detect(text: str) -> str:
    return top_label(local_detector(text, top_k=1, truncation=True))


def translation_service(text: str, target_language: str = "en") -> dict:
    """Detect the language of `text` and translate it into `target_language` if a local model exists."""
    source = local_detect(text)
    if source == target_language:
        return {"source_language": source, "text": text, "translated": False}
    if (source, target_language) not in MARIAN_MODELS:
        return {"source_language": source, "text": text, "translated": False,
                "note": f"no local model for {source} -> {target_language}"}
    tokenizer, model = marian(source, target_language)
    generated = model.generate(**tokenizer([text], return_tensors="pt", padding=True))
    return {"source_language": source, "text": tokenizer.decode(generated[0], skip_special_tokens=True), "translated": True}


if pipeline is None:
    print("⚠️ Skipped: install the hf group first (see the cell above).")
else:
    for expected, text in TEST_MESSAGES:
        result = translation_service(text, target_language="en")
        agree = "✅" if result["source_language"] == llm_detected[text] else "❌"
        print(f"{agree} local {result['source_language']} vs LLM {llm_detected[text]} | {text}")
        print(f"      -> {result['text'] if result['translated'] else result.get('note', 'already English')}\n")

    reply_de = translation_service("Reception is open 24 hours a day.", target_language="de")
    print(f"English -> German: {reply_de['text']}")

**Reading the output:** the ✅/❌ column compares the local detector with the LLM's `language_code` from Part 1.
The French and Spanish messages print "no local model": with only the de↔en pair loaded, the local service
cannot translate them. That is the trade-off in one line: supporting another language locally means another
model download per direction.

---
## 📝 Summary

### 1. Structured detection and translation
- `with_structured_output(GuestMessage)` returns the language code and the English text as typed fields: no
  parsing of model prose.

### 2. Guest attributes in state
- `guest_language`, `sentiment` and `polarity` are saved per `thread_id` by the checkpointer and recomputed on
  every message, so guests never overwrite each other.

### 3. Lexicon sentiment as a code node
- `PatternAnalyzer` averages per-word scores from a bundled English lexicon; `sentiment_analysis` is a plain
  node, and the coordinator adds `DEESCALATE` only when `state["sentiment"] == "negative"`.

### 4. One composed graph
- `translate_in → sentiment_analysis → conv_assistant … → translate_out`: the agents in between did not change.
  The edges, the node order, the reply language, the Spanish booking and the apology were checked in code.

### 5. Limits and alternatives
- The lexicon is English-only and word-level: raw German scores nothing and French only by cognate luck, while
  the translated text scores correctly; sarcasm still fools it. MarianMT + XLM-RoBERTa (optional) run offline,
  at the cost of one model per language pair.

### Next Steps
- `09_hotel_system_wide_error_handling.ipynb`: error handling and recovery across the whole system